# M2 STOP GATE — verify the lakehouse end to end

Reads `bronze.hello` (written by the Dagster asset `platform_orchestration.assets.hello`) straight off SeaweedFS via DuckDB's `iceberg` + `httpfs` extensions, bypassing the REST catalog entirely — this is the same path a data analyst would use to sanity-check a table without a SQL engine attached to Lakekeeper.

Run `dagster asset materialize --select bronze/hello -m platform_orchestration.definitions` (inside the `dagster` container) first, then fetch the table's current `metadata-location` from Lakekeeper's REST API, then run this notebook.

In [1]:
import os
import duckdb
import requests

LAKEKEEPER_URL = "http://lakekeeper:8181"
PROJECT_ID = "00000000-0000-0000-0000-000000000000"

warehouses = requests.get(
    f"{LAKEKEEPER_URL}/management/v1/warehouse", params={"project-id": PROJECT_ID}
).json()["warehouses"]
warehouse_id = next(w["id"] for w in warehouses if w["name"] == "poc")

table = requests.get(
    f"{LAKEKEEPER_URL}/catalog/v1/{warehouse_id}/namespaces/bronze/tables/hello"
).json()
metadata_location = table["metadata-location"]
metadata_location

's3://warehouse/iceberg/019fd433-1a8a-7720-a452-36e9b848c022/019fd433-1b48-7112-9545-e1d5fe292b53/metadata/00002-019fd6be-d3ef-7a10-8932-0b14b84cd35f.gz.metadata.json'

In [2]:
# Exact SET s3_* settings that work against SeaweedFS (MASTER_PROMPT STOP GATE 2
# asks for these explicitly — path-style addressing is the load-bearing one,
# per docs/adr/0001; region is arbitrary since SeaweedFS ignores AWS region
# validation but DuckDB still requires *something* set.
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL iceberg; LOAD iceberg;")
con.execute("SET s3_endpoint='seaweedfs:8333';")
con.execute("SET s3_use_ssl=false;")
con.execute("SET s3_url_style='path';")
con.execute("SET s3_region='local-01';")
con.execute(f"SET s3_access_key_id='{os.environ.get('SEAWEEDFS_ACCESS_KEY', 'platform')}';")
con.execute(
    f"SET s3_secret_access_key='{os.environ.get('SEAWEEDFS_SECRET_KEY', 'platform-dev-secret-please-rotate')}';"
)

In [3]:
con.execute(f"SELECT * FROM iceberg_scan('{metadata_location}')").fetchdf()

,id,message
0,1,hello
1,2,from
2,3,dagster
